# 10 — Query-Level Error Analysis (M8)

Runs `scripts/error_analysis.py`: classifies every (query, relevant document) pair from the real test qrels into 6 diagnostic categories using real per-model ranks (`src/biomedical_ir/error_analysis.py`).

In [ ]:
# If running on Colab, clone the repo and install deps. Skipped automatically
# when already inside a local checkout (REPO_ROOT / 'src' already importable).
import os, sys, subprocess

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    if not os.path.exists('biomedical-hybrid-ir'):
        subprocess.run(['git', 'clone', 'https://github.com/Arungharami/biomedical-hybrid-ir'], check=True)
    os.chdir('biomedical-hybrid-ir')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], check=True)

sys.path.insert(0, os.path.join(os.getcwd(), 'src'))
print('cwd:', os.getcwd())

In [ ]:
import time
start = time.perf_counter()
result = subprocess.run([sys.executable, 'scripts/error_analysis.py'], cwd=os.getcwd())
print(f'\nexit code: {result.returncode}, elapsed: {time.perf_counter()-start:.1f}s')
assert result.returncode == 0, 'Script failed -- see output above.'

## Counts per category

In [ ]:
import json

payload = json.load(open('results/error-analysis/error_analysis.json'))
for category, count in payload['total_examples_found_per_category'].items():
    print(f'{category:35s} {count:4d}')

## A concrete example: lexical precision beats semantic drift

The single-word query "salmon" is ranked #2 by BM25 (exact match) but entirely missed by MedCPT — real evidence that dense retrieval doesn't uniformly win, even though it wins in aggregate (see notebook 09).

In [ ]:
examples = payload['categories']['bm25_wins_medcpt_loses']
salmon_example = next((e for e in examples if e['query'].lower() == 'salmon'), examples[0])
print('Query:', salmon_example['query'])
print('Relevant doc:', salmon_example['doc_id'], '-', salmon_example['doc_title'])
print('Ranks:', salmon_example['ranks'])

See `results/error-analysis/error_analysis.md` for the full set of examples across all six categories.